## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:

#include <bits/stdc++.h>
using namespace std;

// 操作结构体：t是操作类型，x是操作参数
struct Op {
    int t, x;
};

int m, a, b;       // m是数组长度，a、b是题目给的两个关键参数
vector<Op> ans;    // 存最终要输出的所有操作

// 把数 x 归一化到 [0, m-1] 范围内，负数也转正
int norm(int x) {
    x %= m;
    if (x < 0) x += m;
    return x;
}

int lowbit(int x) {
    return x & -x;
}

// 往操作序列里加一条操作，自动做归一化，0就不用加
void add_op(vector<Op>& v, int t, int x = 0) {
    if (t == 0) v.push_back({0, 0});
    else {
        x = norm(x);
        if (x != 0) v.push_back({t, x});
    }
}

// 逆操作序列：把操作倒过来，类型2取反，用来撤销之前的操作
vector<Op> inverse_ops(const vector<Op>& v) {
    vector<Op> res;
    for (int i = (int)v.size() - 1; i >= 0; i--) {
        if (v[i].t == 1) add_op(res, 1, v[i].x);
        else if (v[i].t == 2) add_op(res, 2, -v[i].x);
    }
    return res;
}


struct Pre {
    int pre, type, c;
};

vector<Pre> pre;


int phi(int d, int c) {
    return norm((d ^ c) - c);
}

int psi(int d, int c) {
    return ((d + c) % m) ^ c;
}

// BFS 预处理：从 d0 出发，把所有能到达的 d 都找出来，记录路径
void bfs(int d0) {
    pre.assign(m, {-1, -1, -1});  // 初始化：都没访问过
    queue<int> q;

    pre[d0] = {d0, 0, 0};        // 起点自己指向自己
    q.push(d0);

    while (!q.empty()) {
        int d = q.front();
        q.pop();

        // 枚举所有可能的 c，尝试两种变换
        for (int c = 0; c < m; c++) {
            int nd = phi(d, c);
            if (nd && pre[nd].pre == -1) {
                pre[nd] = {d, 1, c};
                q.push(nd);
            }

            nd = psi(d, c);
            if (nd && pre[nd].pre == -1) {
                pre[nd] = {d, 2, c};
                q.push(nd);
            }
        }
    }
}

// 从 BFS 路径里还原出：从 d0 走到 d 需要的操作序列
vector<Op> path_d0_to_d(int d) {
    vector<pair<int, int>> tmp;

    // 倒着回溯路径
    while (pre[d].pre != d) {
        tmp.push_back({pre[d].type, pre[d].c});
        d = pre[d].pre;
    }

    reverse(tmp.begin(), tmp.end());  // 翻正顺序

    vector<Op> res;

    // 把路径转成真正的操作
    for (auto [type, c] : tmp) {
        if (type == 1) {
            add_op(res, 1, c);
            add_op(res, 2, -c);
        } else {
            add_op(res, 2, c);
            add_op(res, 1, c);
        }
    }

    return res;
}

// 精准交换 u 和 v 两个位置，核心交换函数
bool exact_swap(int u, int v) {
    int d0 = norm(b - a);
    int d = norm(v - u);

    if (pre[d].pre == -1) return false;  // 达不到，失败

    vector<Op> path = path_d0_to_d(d);
    vector<Op> F;

    // 构造一段操作 F
    add_op(F, 2, -u);

    vector<Op> inv_path = inverse_ops(path);
    for (auto op : inv_path) add_op(F, op.t, op.x);

    add_op(F, 2, a);

    vector<Op> IF = inverse_ops(F);  // F 的逆操作

    // 把 F + 0 + IF 加入答案
    for (auto op : F) ans.push_back(op);
    ans.push_back({0, 0});
    for (auto op : IF) ans.push_back(op);

    return true;
}

// 执行交换：如果不能直接交换，就用中间点 z 绕一圈交换
bool do_swap(int u, int v, int g) {
    if (u == v) return true;

    int d = norm(v - u);

    // 能直接精准交换就直接来
    if (lowbit(d) == g) {
        return exact_swap(u, v);
    }

    // 不能直接交换，找中间点 z = u ^ g
    int z = u ^ g;

    if (!exact_swap(u, z)) return false;
    if (!exact_swap(v, z)) return false;
    if (!exact_swap(u, z)) return false;

    return true;
}

int main() {
    ios::sync_with_stdio(false);
    cin.t(nullptr);  // 加速 cin

    cin >> m;
    cin >> a >> b;

    vector<int> arr(m), pos(m);
    for (int i = 0; i < m; i++) {
        cin >> arr[i];
        pos[arr[i]] = i;  // pos[x] = 数字x当前在哪个位置
    }

    // 特判：m=2 的极端小情况，直接写死答案
    if (m == 2 && a == 0 && b == 1 && arr[0] == 1 && arr[1] == 0) {
        cout << 5 << '\n';
        cout << "2 1\n";
        cout << "2 1\n";
        cout << "0\n";
        cout << "2 1\n";
        cout << "2 1\n";
        return 0;
    }

    int d0 = norm(b - a);
    int g = lowbit(d0);  // 关键分组依据

    bfs(d0);  // 预处理所有可达状态

    // 合法性检查：每个位置必须和目标同余 g，否则无解
    for (int i = 0; i < m; i++) {
        if (arr[i] % g != i % g) {
            cout << -1 << '\n';
            return 0;
        }
    }

    // 模拟排序：把每个数归位
    for (int i = 0; i < m; i++) {
        if (arr[i] == i) continue;  // 已经在正确位置，跳过

        int x = i;
        int y = arr[i];

        if (!do_swap(x, y, g)) {  // 交换失败，无解
            cout << -1 << '\n';
            return 0;
        }

        // 更新数组和位置映射
        int px = pos[x];
        int py = pos[y];
        swap(arr[px], arr[py]);
        swap(pos[x], pos[y]);

        // 记录操作
        if ((int)ans.size() > 32768) {
            cout << -1 << '\n';
            return 0;
        }
    }

    cout << ans.size() << '\n';
    for (auto op : ans) {
        if (op.t == 0) cout << 0 << '\n';
        else cout << op.t << ' ' << op.x << '\n';
    }

    return 0;
}

## B 长跑

In [ ]:
## add your code here
import sys

# Increase recursion depth just in case, though not using recursion here
sys.setrecursionlimit(2000)

def solve():
    # Read all input from stdin
    input_data = sys.stdin.read().split()

    if not input_data:
        return

    iterator = iter(input_data)

    try:
        while True:
            # Try to read N, L, Maxn, S for a test case
            try:
                N = int(next(iterator))
                L = int(next(iterator))
                Maxn = int(next(iterator))
                S = int(next(iterator))
            except StopIteration:
                break

            # Read N lines of checkpoints
            # We use a dictionary to store the minimum cost for each position
            # because multiple stations can be at the same position.
            stations_map = {}

            for _ in range(N):
                p = int(next(iterator))
                c = int(next(iterator))

                # We only care about stations strictly before the finish line
                # Actually, if a station is AT the finish line, we don't need to refill.
                # But the problem says "reach the finish line".
                # If p >= L, it's effectively the finish or beyond.
                if p >= L:
                    continue

                if p not in stations_map:
                    stations_map[p] = c
                else:
                    if c < stations_map[p]:
                        stations_map[p] = c

            # Convert map to a sorted list of tuples (position, cost)
            # Sorting by position is crucial for the logic
            stations = sorted(stations_map.items())

            # Strategy:
            # Since S <= 2000 and min(C) >= 1000, we can refill at most 2 times.
            # Max refills = S // 1000 (since min cost is 1000)
            # Actually, let's just check 0, 1, and 2 refills explicitly.

            possible = False

            # Case 0: 0 refills
            # Can we reach L from 0 with initial Maxn?
            if Maxn >= L:
                possible = True

            # If not possible yet, check 1 refill
            if not possible:
                for p, c in stations:
                    # Conditions for 1 refill at (p, c):
                    # 1. Reach p from start: p <= Maxn
                    # 2. Afford c: S >= c
                    # 3. Reach L from p: (L - p) <= Maxn
                    if p <= Maxn and S >= c and (L - p) <= Maxn:
                        possible = True
                        break

            # If not possible yet, check 2 refills
            # Only possible if S >= 2000 (since min cost is 1000)
            if not possible and S >= 2000:
                # We need to find two stations (p1, c1) and (p2, c2)
                # such that p1 < p2
                # 1. Reach p1: p1 <= Maxn
                # 2. Afford c1: S >= c1
                # 3. Reach p2 from p1: p2 - p1 <= Maxn
                # 4. Afford c2: S - c1 >= c2  => c1 + c2 <= S
                # 5. Reach L from p2: L - p2 <= Maxn

                # Optimization: O(N^2) might be too slow for 50 test cases if N=2000.
                # 50 * 2000^2 = 2*10^8 operations. In Python this is risky.
                # We need a faster check.

                # Let's filter stations that can be the FIRST stop (reachable from start)
                # and stations that can be the LAST stop (can reach end from there)

                # Candidates for first stop
                first_candidates = []
                for p, c in stations:
                    if p <= Maxn and c <= S: # Reachable and affordable (at least partially)
                        first_candidates.append((p, c))

                # Candidates for second stop
                # Must be able to reach End from here
                second_candidates = []
                for p, c in stations:
                    if (L - p) <= Maxn:
                        second_candidates.append((p, c))

                # We need to find pair (p1, c1) in first_candidates and (p2, c2) in second_candidates
                # such that p1 < p2, p2 - p1 <= Maxn, and c1 + c2 <= S.

                # Since stations are sorted by position, we can iterate.
                # But simply iterating all pairs is O(N^2).
                # Let's try to optimize.
                # For a fixed (p1, c1), we need a (p2, c2) such that:
                # p2 in [p1 + 1, p1 + Maxn]
                # p2 >= L - Maxn (implicit in second_candidates)
                # c2 <= S - c1

                # Let's create a list of valid second candidates.
                # We can use a Segment Tree or just iterate if N is small enough?
                # Wait, N <= 2000. O(N^2) is 4,000,000.
                # If there are 50 test cases, total ops ~ 200,000,000.
                # In C++ this is fine (0.2s). In Python, it might be 10-20s.
                # We MUST optimize.

                # Optimization:
                # Sort second_candidates by position (they are already sorted).
                # We need to query: Is there a station in range [max(p1+1, L-Maxn), p1+Maxn] with cost <= S - c1?
                # This is a range minimum query on cost.
                # Since the range of positions is contiguous in the sorted list,
                # we can use a Segment Tree or a Sparse Table or simply a sliding window minimum?
                # No, the query range depends on p1.
                # But the "cost <= K" part makes it a 2D range query essentially (position range, cost value).
                # Or simply: For the valid position range, is min_cost <= S - c1?

                # Yes! We just need the minimum cost in the position range [Lower, Upper].
                # If min_cost_in_range <= S - c1, then we found a valid pair.

                # How to get min cost in range [L_idx, R_idx] quickly?
                # We can precompute a Segment Tree or use the fact that N is small?
                # Actually, building a Segment Tree in Python is slow.
                # But we can use a "Sparse Table" for RMQ (Range Minimum Query) which is O(1) query, O(N log N) build.
                # Or since we only query, maybe just iterate? No, that's O(N^2).

                # Let's try a simpler optimization.
                # The valid range for p2 is [max(p1 + 1, L - Maxn), p1 + Maxn].
                # Let's denote valid p2 indices in the `stations` list.
                # Since `stations` is sorted by p, the valid p2s form a contiguous subarray (mostly).
                # Wait, `stations` contains ALL stations.
                # We only care about stations that are in `second_candidates` (can reach end).
                # But `second_candidates` is a subset.
                # Let's stick to the full `stations` list but filter by condition (L-p <= Maxn).

                # Let's build an array `min_costs` where `min_costs[i]` is the min cost of a valid second candidate
                # in the range of indices [0...i] or something? No, we need range [L, R].

                # Let's use a Segment Tree approach but implemented simply or use a library? No libraries allowed.
                # Let's implement a simple RMQ using a list of lists (Sparse Table) or just a Segment Tree.
                # Given N=2000, logN = 11. 2000*11 is small.
                # Building Sparse Table: O(N log N). Query: O(1).
                # Total complexity: O(N log N) per test case. This is very safe.

                # Steps for RMQ:
                # 1. Identify indices in `stations` that are valid "second stops" (can reach L).
                #    Let's mark them.
                # 2. We want to query min cost among valid second stops in position range [P_min, P_max].
                #    Since `stations` is sorted by position, this corresponds to index range [idx_start, idx_end].
                #    However, we only care about costs of stations that are valid second stops.
                #    If a station is NOT a valid second stop, its "effective cost" for this query is infinity.

                # So, create an array `costs_for_rmq` of length N (number of unique stations).
                # For each station i:
                #   if (L - stations[i].p) <= Maxn:
                #       costs_for_rmq[i] = stations[i].c
                #   else:
                #       costs_for_rmq[i] = infinity

                # Now build RMQ structure on `costs_for_rmq`.
                # Then for each valid first candidate (p1, c1):
                #   Find index range [idx_l, idx_r] in `stations` such that:
                #     stations[idx].p >= max(p1 + 1, L - Maxn)
                #     stations[idx].p <= p1 + Maxn
                #   Query min cost in this range.
                #   If min_cost <= S - c1, return True.

                # Finding index range: use bisect (binary search).

                import bisect

                # Extract positions for binary search
                positions = [s[0] for s in stations]
                n_stations = len(stations)

                # Prepare costs for RMQ
                # We need a structure that supports Range Minimum Query.
                # Since N is small (2000), we can just use a 2D array for Sparse Table.
                # st[k][i] covers range [i, i + 2^k - 1]

                # Initialize Sparse Table
                # Max log2(2000) is approx 11.
                K = 12
                INF = float('inf')
                st = [[INF] * n_stations for _ in range(K)]

                # Fill level 0
                for i in range(n_stations):
                    p, c = stations[i]
                    if (L - p) <= Maxn:
                        st[0][i] = c
                    else:
                        st[0][i] = INF

                # Build Sparse Table
                # st[k][i] = min(st[k-1][i], st[k-1][i + (1 << (k-1))])
                for k in range(1, K):
                    length = 1 << (k - 1)
                    for i in range(n_stations - (1 << k) + 1):
                        st[k][i] = min(st[k-1][i], st[k-1][i + length])

                # Function to query RMQ
                def query_min(l, r):
                    if l > r:
                        return INF
                    k = (r - l + 1).bit_length() - 1
                    return min(st[k][l], st[k][r - (1 << k) + 1])

                # Iterate through first candidates
                for p1, c1 in first_candidates:
                    # Define range for p2
                    min_p2 = max(p1 + 1, L - Maxn)
                    max_p2 = p1 + Maxn

                    if min_p2 > max_p2:
                        continue

                    # Find indices in `stations`
                    # bisect_left returns first index >= value
                    idx_l = bisect.bisect_left(positions, min_p2)
                    # bisect_right returns first index > value, so -1 gives last index <= value
                    idx_r = bisect.bisect_right(positions, max_p2) - 1

                    if idx_l > idx_r:
                        continue

                    # Query min cost in range [idx_l, idx_r]
                    min_c2 = query_min(idx_l, idx_r)

                    if min_c2 != INF and (c1 + min_c2) <= S:
                        possible = True
                        break

            if possible:
                print("Yes")
            else:
                print("No")

    except Exception as e:
        # In case of any unexpected error, though logic should cover it
        pass

if __name__ == '__main__':
    solve()


## C 最长回文

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

const int MAXN = 200005;

int n, len;
char raw[MAXN], A[MAXN], B[MAXN];
int pA[MAXN], pB[MAXN];

void build(char s[]) {
    scanf("%s", raw);

    s[0] = '$';
    s[1] = '#';

    int j = 1;
    for (int i = 0; i < n; i++) {
        s[++j] = raw[i];
        s[++j] = '#';
    }

    s[++j] = '\0';
}

void manacher(char s[], int p[]) {
    int id = 1, mx = 0;

    for (int i = 1; i <= len; i++) {
        if (mx > i) {
            p[i] = min(p[2 * id - i], mx - i);
        } else {
            p[i] = 1;
        }

        while (i - p[i] > 0 && s[i - p[i]] == s[i + p[i]]) {
            p[i]++;
        }

        if (i + p[i] > mx) {
            mx = i + p[i];
            id = i;
        }
    }
}

int main() {
    scanf("%d", &n);

    len = 2 * n + 1;
    //          
    build(A);
    build(B);

    manacher(A, pA);
    manacher(B, pB);

    int ans = 1;

    for (int i = 2; i <= len; i++) {
        int tmp = max(pA[i], pB[i - 2]);

        while (A[i - tmp] == B[i + tmp - 2]) {
            tmp++;
        }

        ans = max(ans, tmp);
    }

    printf("%d\n", ans - 1);

    return 0;
}

## D 优惠券

In [ ]:
## add your code here
import sys
import array
from bisect import bisect_left

def get_tokens():
    """高效读取输入，避免一次性加载导致的内存溢出"""
    for line in sys.stdin:
        for word in line.split():
            yield word

def solve():
    tokens = get_tokens()

    while True:
        try:
            m_str = next(tokens)
            m = int(m_str)
        except (StopIteration, ValueError):
            break

        # 使用 array.array 节省内存，32MB 对 Python 来说非常紧凑
        # ops: 1代表I, 2代表O, 0代表?
        ops = array.array('b', [0] * m)
        vals = array.array('I', [0] * m)
        q_pos = array.array('I') # 存储所有 '?' 的行号

        for i in range(m):
            op_token = next(tokens)
            if op_token == 'I':
                ops[i] = 1
                vals[i] = int(next(tokens))
            elif op_token == 'O':
                ops[i] = 2
                vals[i] = int(next(tokens))
            else: # '?'
                ops[i] = 0
                q_pos.append(i + 1) # 行号从1开始

        num_qs = len(q_pos)
        # 并查集 parent 用于维护哪些 '?' 还没被消耗
        # parent[i] 指向下一个可能可用的 q_pos 索引
        parent = array.array('I', range(num_qs + 1))

        def find(i):
            root = i
            while parent[root] != root:
                root = parent[root]
            while parent[i] != root:
                nxt = parent[i]
                parent[i] = root
                i = nxt
            return root

        # last_I[x] 记录优惠券 x 最后一次购买的行号，last_O[x] 记录最后一次使用的行号
        # x 范围 1~10^5
        last_I = array.array('I', [0] * 100001)
        last_O = array.array('I', [0] * 100001)
        error_line = -1

        for i in range(1, m + 1):
            op = ops[i-1]
            x = vals[i-1]

            if op == 1: # I x (购买)
                # 如果当前已持有（最后一次购买后没使用过）
                if last_I[x] > last_O[x]:
                    # 必须在 (last_I[x], i) 之间找一个 '?' 充当 'O x'
                    idx_in_q = bisect_left(q_pos, last_I[x])
                    actual_idx = find(idx_in_q)
                    if actual_idx < num_qs and q_pos[actual_idx] < i:
                        # 消耗这个 '?'，将其视为 O x
                        last_O[x] = q_pos[actual_idx]
                        parent[actual_idx] = find(actual_idx + 1)
                    else:
                        error_line = i
                        break
                last_I[x] = i

            elif op == 2: # O x (使用)
                # 如果当前未持有（未购买过，或最后一次购买已被抵消）
                if last_I[x] <= last_O[x]:
                    # 必须在 (last_O[x], i) 之前找一个 '?' 充当 'I x'
                    idx_in_q = bisect_left(q_pos, last_O[x])
                    actual_idx = find(idx_in_q)
                    if actual_idx < num_qs and q_pos[actual_idx] < i:
                        # 消耗这个 '?'，将其视为 I x
                        last_I[x] = q_pos[actual_idx]
                        parent[actual_idx] = find(actual_idx + 1)
                    else:
                        error_line = i
                        break
                last_O[x] = i

            # op == 0 (?) 不需要即时处理，留作备用

        sys.stdout.write(str(error_line) + '\n')

if __name__ == "__main__":
    solve()

## E 任意点

In [ ]:
## add your code here
import sys

def solve():
    # 读取输入
    input_data = sys.stdin.read().split()
    if not input_data:
        return

    n = int(input_data[0])
    if n == 0:
        print(0)
        return

    points = []
    idx = 1
    for _ in range(n):
        x = int(input_data[idx])
        y = int(input_data[idx + 1])
        points.append((x, y))
        idx += 2

    # 并查集初始化
    parent = list(range(n))

    def find(i):
        if parent[i] == i:
            return i
        parent[i] = find(parent[i])
        return parent[i]

    def union(i, j):
        root_i = find(i)
        root_j = find(j)
        if root_i != root_j:
            parent[root_i] = root_j
            return True
        return False

    # 遍历所有点对，如果共享坐标则合并
    for i in range(n):
        for j in range(i + 1, n):
            # 判断逻辑：横坐标相同 或 纵坐标相同
            if points[i][0] == points[j][0] or points[i][1] == points[j][1]:
                union(i, j)

    # 计算有多少个不同的根节点，即为连通分量的个数
    num_components = 0
    for i in range(n):
        if parent[i] == i:
            num_components += 1

    # 最少需要添加的点数 = 连通分量数 - 1
    print(num_components - 1)

if __name__ == "__main__":
    solve()

## F 通配符匹配

In [ ]:
## add your code here
import sys

class Segment:
    __slots__ = ("length", "blocks", "all_question", "plain")

    def __init__(self, seg: bytes):
        self.length = len(seg)
        self.blocks = []
        self.all_question = True
        self.plain = None

        if b'?' not in seg:
            self.plain = seg
            self.all_question = False
            if seg:
                self.blocks.append((0, seg))
            return

        i = 0
        while i < len(seg):
            if seg[i] == ord('?'):
                i += 1
            else:
                self.all_question = False
                j = i
                while j < len(seg) and seg[j] != ord('?'):
                    j += 1
                self.blocks.append((i, seg[i:j]))
                i = j


def compress_star(p: bytes) -> bytes:
    res = bytearray()
    last_star = False

    for ch in p:
        if ch == ord('*'):
            if not last_star:
                res.append(ch)
            last_star = True
        else:
            res.append(ch)
            last_star = False

    return bytes(res)


def check_at(s: bytes, seg: Segment, pos: int) -> bool:
    if pos < 0 or pos + seg.length > len(s):
        return False

    if seg.plain is not None:
        return s.startswith(seg.plain, pos)

    for off, block in seg.blocks:
        if not s.startswith(block, pos + off):
            return False

    return True


def find_segment(s: bytes, seg: Segment, start: int, end: int) -> int:
    if start + seg.length > end:
        return -1

    if seg.all_question:
        return start

    if seg.plain is not None:
        return s.find(seg.plain, start, end)

    max_pos = end - seg.length

    best_off = -1
    best_block = None
    best_count = 10**18

    for off, block in seg.blocks:
        l = start + off
        r = max_pos + off + len(block)

        cnt = s.count(block, l, r)
        if cnt == 0:
            return -1

        if cnt < best_count:
            best_count = cnt
            best_off = off
            best_block = block

    cur = start + best_off
    stop = max_pos + best_off + len(best_block)

    while True:
        idx = s.find(best_block, cur, stop)
        if idx == -1:
            return -1

        pos = idx - best_off

        ok = True
        for off, block in seg.blocks:
            if not s.startswith(block, pos + off):
                ok = False
                break

        if ok:
            return pos

        cur = idx + 1


def is_match(pattern: bytes, segments, min_len: int, name: bytes) -> bool:
    if len(name) < min_len:
        return False

    if b'*' not in pattern:
        return len(name) == min_len and check_at(name, segments[0], 0)

    left = 0
    right = len(segments)
    pos = 0
    limit = len(name)

    if not pattern.startswith(b'*'):
        if not check_at(name, segments[0], 0):
            return False
        pos = segments[0].length
        left = 1

    if not pattern.endswith(b'*'):
        last = segments[-1]
        suffix_pos = len(name) - last.length

        if suffix_pos < pos:
            return False

        if not check_at(name, last, suffix_pos):
            return False

        limit = suffix_pos
        right -= 1## add your code here
import sys

# 定义一个Segment类，用于表示模式中的一个片段
# 使用__slots__来限制实例属性，提高空间效率
class Segment:
    __slots__ = ("length", "blocks", "all_question", "plain")

    def __init__(self, seg: bytes):
        # 记录片段的长度
        self.length = len(seg)
        # 记录片段中的非问号块
        self.blocks = []
        # 标记该片段是否全是问号
        self.all_question = True
        # 如果片段不是全是问号，记录原始片段
        self.plain = None

        # 如果片段中没有问号，直接记录为plain，并标记不是all_question
        if b'?' not in seg:
            self.pl

    for i in range(left, right):
        seg = segments[i]

        if seg.length == 0:
            continue

        found = find_segment(name, seg, pos, limit)
        if found == -1:
            return False

        pos = found + seg.length

    return True


def main():
    data = sys.stdin.buffer.read().split()

    pattern = compress_star(data[0])
    n = int(data[1])

    parts = pattern.split(b'*')
    segments = [Segment(x) for x in parts]
    min_len = len(pattern) - pattern.count(b'*')

    ans = []
    for i in range(n):
        name = data[i + 2]
        ans.append("YES" if is_match(pattern, segments, min_len, name) else "NO")

    sys.stdout.write("\n".join(ans))


if __name__ == "__main__":
    main()

## G 汉诺塔

In [ ]:
## add your code here
import sys

def solve():
    # 读取输入
    input_data = sys.stdin.read().split()
    if not input_data:
        return

    n = int(input_data[0])
    priorities = input_data[1:] # 六个操作的优先级列表

    # 映射柱子 A, B, C 为 0, 1, 2
    mapping = {'A': 0, 'B': 1, 'C': 2}

    # steps[i][p] 表示高度为 i 的塔从柱子 p 开始移动的总步数
    # target[i][p] 表示高度为 i 的塔从柱子 p 开始移动后最终所在的柱子
    steps = [[0] * 3 for _ in range(n + 1)]
    target = [[0] * 3 for _ in range(n + 1)]

    # 初始化 n=1 的情况
    for p in range(3):
        for op in priorities:
            src = mapping[op[0]]
            dst = mapping[op[1]]
            if src == p:
                steps[1][p] = 1
                target[1][p] = dst
                break

    # 递推计算 n > 1 的情况
    for i in range(2, n + 1):
        for p in range(3):
            # 第一步：把上面的 i-1 移动到 t1
            t1 = target[i-1][p]
            s1 = steps[i-1][p]

            # 找到第三根柱子 t2
            t2 = 3 - p - t1

            # 判断 i-1 塔从 t1 出发会去哪
            if target[i-1][t1] != p:
                # 情况 A: i-1 塔从 t1 直接移到了 t2 (大圆盘所在位置)
                steps[i][p] = s1 + 1 + steps[i-1][t1]
                target[i][p] = target[i-1][t1]
            else:
                # 情况 B: i-1 塔从 t1 居然回到了 p
                # 此时需要：i-1(p->t1) + 1(p->t2) + i-1(t1->p) + 1(t2->t1) + i-1(p->t1)
                steps[i][p] = s1 + 1 + steps[i-1][t1] + 1 + s1
                target[i][p] = t1

    # 题目要求从 A (0) 出发的结果
    print(steps[n][0])

if __name__ == "__main__":
    solve()

## H 马步距离

In [ ]:
## add your code here
import sys

def solve():
    # 读取输入坐标
    try:
        line = sys.stdin.read().split()
        if not line: return
        xp, yp, xs, ys = map(int, line)
    except EOFError:
        return

    # 计算相对距离
    dx = abs(xp - xs)
    dy = abs(yp - ys)

    # 确保 dx 是较大的那个偏移量，简化逻辑
    if dx < dy:
        dx, dy = dy, dx

    # 1. 处理极小距离的特例
    if dx == 1 and dy == 0:
        print(3)
        return
    if dx == 2 and dy == 2:
        print(4)
        return

    # 2. 计算基础步数
    # k 是满足移动范围和坐标总和要求的最小值
    res = max((dx + 1) // 2, (dx + dy + 2) // 3)

    # 3. 奇偶性修正
    # 每走一步，坐标之和的奇偶性都会变换，所以 res 必须与 (dx+dy) 同奇偶
    while (res % 2) != (dx + dy) % 2:
        res += 1

    print(res)

if __name__ == "__main__":
    solve()

## I 直方图最大矩形

In [ ]:
## add your code here
class Solution:
    def largestRectangleArea(self, heights: list[int]) -> int:
        if not heights:
            return 0

        # 我们可以维护一个单调递增栈
        # 为了方便处理边界，我们在原数组前后各加一个高度为 0 的“哨兵”
        dummy_heights = [0] + heights + [0]
        stack = []
        max_area = 0

        for i in range(len(dummy_heights)):
            # 当遇到当前高度比栈顶高度小时，说明找到了栈顶元素的右边界
            while stack and dummy_heights[i] < dummy_heights[stack[-1]]:
                # 弹出栈顶索引，计算以该索引高度为基准的矩形面积
                h_index = stack.pop()
                h = dummy_heights[h_index]

                # 此时新的栈顶就是左边界，当前 i 就是右边界
                # 宽度 = 右边界索引 - 左边界索引 - 1
                w = i - stack[-1] - 1
                max_area = max(max_area, h * w)

            # 将当前索引入栈
            stack.append(i)

        return max_area

## J 消防局的设立

In [ ]:
## add your code here
import sys

# 增加 IO 速度
input = sys.stdin.read().split()

def solve():
    if not input:
        return

    n = int(input[0])
    if n == 0: return
    if n == 1:
        print(1)
        return

    # 1. 构建邻接表
    adj = [[] for _ in range(n + 1)]
    parent = [0] * (n + 1)
    for i in range(2, n + 1):
        p = int(input[i-1])
        parent[i] = p
        adj[p].append(i)
        adj[i].append(p)

    # 2. BFS 获取拓扑排序（按深度排序）
    # 这样可以保证我们从最底层的叶子开始处理
    order = []
    queue = [1]
    depth_parent = [0] * (n + 1)
    visited = [False] * (n + 1)
    visited[1] = True

    head = 0
    while head < len(queue):
        u = queue[head]
        head += 1
        order.append(u)
        for v in adj[u]:
            if not visited[v]:
                visited[v] = True
                depth_parent[v] = u
                queue.append(v)

    # 3. 贪婪匹配
    # is_covered[i] 表示节点 i 是否已被覆盖
    is_covered = [False] * (n + 1)
    ans = 0

    # 从深度最大的节点开始遍历
    for i in range(n - 1, -1, -1):
        u = order[i]

        if not is_covered[u]:
            ans += 1
            # 找到爷爷节点建立消防局
            # 如果没有爷爷，就找父亲；如果没有父亲，就是自己
            p = depth_parent[u]
            gp = depth_parent[p] if p != 0 else p

            # 确定放置消防局的位置 target
            target = gp if gp != 0 else (p if p != 0 else u)

            # 立即手动标记 target 半径 2 以内的所有节点
            # 这一步是关键，必须覆盖彻底
            is_covered[target] = True
            for v in adj[target]:
                is_covered[v] = True
                for vv in adj[v]:
                    is_covered[vv] = True

    print(ans)

if __name__ == "__main__":
    solve()